# AIOps Detection Report

Summarizes record-level PyTorch AIOps detection results across Bronze, Silver, and Gold.

Reports included:

- scored record counts by layer
- PASS / WARN / QUARANTINE counts
- severity distribution
- source/error type distribution
- layer x decision matrix
- top contributing features for WARN and QUARANTINE records
- recent runtime metrics

Input tables:

- `report_aiops_bronze_record_decisions`
- `report_aiops_silver_record_decisions`
- `report_aiops_gold_record_decisions`

In [0]:
from functools import reduce
import json

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalog_name", "hant-catalog")
dbutils.widgets.text("schema_name", "hsl")
dbutils.widgets.text("report_start_date", "")
dbutils.widgets.text("report_end_date", "")
dbutils.widgets.dropdown("persist_report_tables", "true", ["true", "false"])
dbutils.widgets.text("storage_account", "streanmingdatasta")
dbutils.widgets.text("lakehouse_container", "lakehouse")
dbutils.widgets.text("report_base_path", "")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
REPORT_START_DATE = dbutils.widgets.get("report_start_date").strip()
REPORT_END_DATE = dbutils.widgets.get("report_end_date").strip()
PERSIST_REPORT_TABLES = dbutils.widgets.get("persist_report_tables").lower() == "true"
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
LAKEHOUSE_CONTAINER = dbutils.widgets.get("lakehouse_container")
REPORT_BASE_PATH_WIDGET = dbutils.widgets.get("report_base_path").strip()
REPORT_BASE_PATH = REPORT_BASE_PATH_WIDGET or f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog/aiops/reports"

DECISION_TABLES = {
    "bronze": "report_aiops_bronze_record_decisions",
    "silver": "report_aiops_silver_record_decisions",
    "gold": "report_aiops_gold_record_decisions",
}

RUNTIME_TABLE = "report_aiops_runtime_metrics"

REPORT_TABLES = {
    "summary": "report_aiops_detection_summary",
    "decision_matrix": "report_aiops_detection_decision_matrix",
    "severity_source_type": "report_aiops_detection_severity_source_type",
    "top_features": "report_aiops_detection_top_features",
    "runtime": "report_aiops_detection_runtime_summary",
}


def qname(table_name: str) -> str:
    return f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{table_name}"


def table_exists(table_name: str) -> bool:
    try:
        spark.table(qname(table_name)).limit(1).collect()
        return True
    except Exception:
        return False


def save_report(df: DataFrame, table_name: str, partition_cols=None) -> None:
    if not PERSIST_REPORT_TABLES:
        return
    path = f"{REPORT_BASE_PATH.rstrip('/')}/{table_name}"
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    spark.sql(f"DROP TABLE IF EXISTS {qname(table_name)}")
    spark.sql(f"CREATE TABLE {qname(table_name)} USING DELTA LOCATION '{path}'")
    spark.sql(f"REFRESH TABLE {qname(table_name)}")
    print(json.dumps({"saved_table": qname(table_name), "path": path}, default=str))


print(json.dumps({
    "decision_tables": {k: qname(v) for k, v in DECISION_TABLES.items()},
    "report_start_date": REPORT_START_DATE or None,
    "report_end_date": REPORT_END_DATE or None,
    "persist_report_tables": PERSIST_REPORT_TABLES,
    "report_base_path": REPORT_BASE_PATH,
}, indent=2))

{
  "decision_tables": {
    "bronze": "`hant-catalog`.hsl.report_aiops_bronze_record_decisions",
    "silver": "`hant-catalog`.hsl.report_aiops_silver_record_decisions",
    "gold": "`hant-catalog`.hsl.report_aiops_gold_record_decisions"
  },
  "report_start_date": null,
  "report_end_date": null,
  "persist_report_tables": true,
  "report_base_path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/reports"
}


In [0]:
decision_dfs = []
missing_tables = []

for layer, table_name in DECISION_TABLES.items():
    if table_exists(table_name):
        df = spark.table(qname(table_name)).withColumn("layer", F.lit(layer))
        decision_dfs.append(df)
    else:
        missing_tables.append(qname(table_name))

if not decision_dfs:
    raise ValueError(f"No AIOps decision tables found. Missing: {missing_tables}")

aiops_decisions = reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), decision_dfs)

if "ingest_date" in aiops_decisions.columns:
    aiops_decisions = aiops_decisions.withColumn("ingest_date", F.to_date("ingest_date"))
elif "feature_date" in aiops_decisions.columns:
    aiops_decisions = aiops_decisions.withColumn("ingest_date", F.col("feature_date"))
else:
    aiops_decisions = aiops_decisions.withColumn("ingest_date", F.col("scored_date"))

if REPORT_START_DATE:
    aiops_decisions = aiops_decisions.where(F.col("ingest_date") >= F.to_date(F.lit(REPORT_START_DATE)))
if REPORT_END_DATE:
    aiops_decisions = aiops_decisions.where(F.col("ingest_date") <= F.to_date(F.lit(REPORT_END_DATE)))

aiops_decisions = aiops_decisions.cache()

display(aiops_decisions.groupBy("layer", "ingest_date").agg(
    F.count("*").alias("scored_records"),
    F.min("scored_at").alias("first_scored_at"),
    F.max("scored_at").alias("last_scored_at"),
).filter(F.col("ingest_date") == F.current_date()).orderBy("layer", "ingest_date"))

if missing_tables:
    print("Missing decision tables:")
    for table_name in missing_tables:
        print(f"  - {table_name}")

layer,ingest_date,scored_records,first_scored_at,last_scored_at
bronze,2026-04-29,142385,2026-04-29T04:32:24.054807Z,2026-04-29T04:36:50.777789Z
gold,2026-04-29,126240,2026-04-29T05:44:20.814154Z,2026-04-29T05:44:20.814154Z
silver,2026-04-29,128280,2026-04-29T05:23:09.707743Z,2026-04-29T05:23:09.707743Z


## 1. Layer Summary

In [0]:
summary_df = (
    aiops_decisions
    .filter(F.col("ingest_date") == F.current_date())
    .groupBy("layer")
    .agg(
        F.count("*").alias("total_scored_records"),
        F.sum(F.when(F.col("decision_action") == "PASS", 1).otherwise(0)).cast("bigint").alias("pass_records"),
        F.sum(F.when(F.col("decision_action") == "WARN", 1).otherwise(0)).cast("bigint").alias("warning_records"),
        F.sum(F.when(F.col("decision_action") == "QUARANTINE", 1).otherwise(0)).cast("bigint").alias("quarantine_records"),
        F.round(F.avg(F.when(F.col("decision_action") == "PASS", 1.0).otherwise(0.0)) * 100, 2).alias("pass_pct"),
        F.round(F.avg(F.when(F.col("decision_action") == "WARN", 1.0).otherwise(0.0)) * 100, 2).alias("warning_pct"),
        F.round(F.avg(F.when(F.col("decision_action") == "QUARANTINE", 1.0).otherwise(0.0)) * 100, 2).alias("quarantine_pct"),
        F.round(F.avg("reconstruction_error"), 8).alias("avg_reconstruction_error"),
        F.round(F.max("reconstruction_error"), 8).alias("max_reconstruction_error"),
        F.min("scored_at").alias("first_scored_at"),
        F.max("scored_at").alias("last_scored_at"),
    )
    .orderBy("layer")
)

display(summary_df)
save_report(summary_df, REPORT_TABLES["summary"])

layer,total_scored_records,pass_records,warning_records,quarantine_records,pass_pct,warning_pct,quarantine_pct,avg_reconstruction_error,max_reconstruction_error,first_scored_at,last_scored_at
bronze,142385,142385,0,0,100.0,0.0,0.0,1.742E-5,2.068E-5,2026-04-29T04:32:24.054807Z,2026-04-29T04:36:50.777789Z
gold,126240,125784,342,114,99.64,0.27,0.09,0.00272462,0.1128873,2026-04-29T05:44:20.814154Z,2026-04-29T05:44:20.814154Z
silver,128280,116334,9906,2040,90.69,7.72,1.59,255.71271022,27240.72070313,2026-04-29T05:23:09.707743Z,2026-04-29T05:23:09.707743Z


{"saved_table": "`hant-catalog`.hsl.report_aiops_detection_summary", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/reports/report_aiops_detection_summary"}


## 2. Detection Matrix

In [0]:
decision_matrix_df = (
    aiops_decisions
    .filter(F.col("ingest_date") == F.current_date())
    .groupBy("layer", "decision_action", "aiops_severity")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.avg("reconstruction_error"), 8).alias("avg_reconstruction_error"),
        F.round(F.max("reconstruction_error"), 8).alias("max_reconstruction_error"),
        F.round(F.avg("anomaly_threshold"), 8).alias("avg_anomaly_threshold"),
        F.round(F.avg("warning_threshold"), 8).alias("avg_warning_threshold"),
        F.round(F.avg("quarantine_threshold"), 8).alias("avg_quarantine_threshold"),
    )
    .orderBy("layer", "decision_action", "aiops_severity")
)

display(decision_matrix_df)
save_report(decision_matrix_df, REPORT_TABLES["decision_matrix"])

layer,decision_action,aiops_severity,record_count,avg_reconstruction_error,max_reconstruction_error,avg_anomaly_threshold,avg_warning_threshold,avg_quarantine_threshold
bronze,PASS,NORMAL,142385,1.742E-5,2.068E-5,0.02030208,0.02030208,0.06090625
gold,PASS,NORMAL,125784,0.00251645,0.03262944,0.03265914,0.03265914,0.09797742
gold,QUARANTINE,CRITICAL,114,0.10798192,0.1128873,0.03265914,0.03265914,0.09797742
gold,WARN,WARNING,342,0.04420214,0.07028722,0.03265914,0.03265914,0.09797742
silver,PASS,NORMAL,116334,0.03049214,0.07382157,0.07382312,0.07382312,0.22146935
silver,QUARANTINE,CRITICAL,2040,16077.56680125,27240.72070313,0.07382312,0.07382312,0.22146935
silver,WARN,WARNING,9906,0.10528163,0.22080296,0.07382312,0.07382312,0.22146935


{"saved_table": "`hant-catalog`.hsl.report_aiops_detection_decision_matrix", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/reports/report_aiops_detection_decision_matrix"}


## 3. Error Type And Severity

In [0]:
severity_source_type_df = (
    aiops_decisions
    .filter(F.col("ingest_date") == F.current_date())
    .groupBy("layer", "primary_source_type", "decision_action", "aiops_severity")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.avg("reconstruction_error"), 8).alias("avg_reconstruction_error"),
        F.round(F.max("reconstruction_error"), 8).alias("max_reconstruction_error"),
    )
    .orderBy("layer", "primary_source_type", "decision_action", F.desc("record_count"))
)

display(severity_source_type_df)
save_report(severity_source_type_df, REPORT_TABLES["severity_source_type"])

layer,primary_source_type,decision_action,aiops_severity,record_count,avg_reconstruction_error,max_reconstruction_error
bronze,timeliness,PASS,NORMAL,142385,1.742E-5,2.068E-5
gold,validity,PASS,NORMAL,125784,0.00251645,0.03262944
gold,validity,QUARANTINE,CRITICAL,114,0.10798192,0.1128873
gold,validity,WARN,WARNING,342,0.04420214,0.07028722
silver,timeliness,PASS,NORMAL,154,0.01294548,0.01625442
silver,validity,PASS,NORMAL,116180,0.0305154,0.07382157
silver,validity,QUARANTINE,CRITICAL,2040,16077.56680125,27240.72070313
silver,validity,WARN,WARNING,9906,0.10528163,0.22080296


{"saved_table": "`hant-catalog`.hsl.report_aiops_detection_severity_source_type", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/reports/report_aiops_detection_severity_source_type"}


## 4. Warning And Quarantine Detail

In [0]:
warning_quarantine_df = (
    aiops_decisions
    .filter(F.col("ingest_date") == F.current_date())
    .where(F.col("decision_action").isin("WARN", "QUARANTINE"))
    .select(
        "layer",
        "record_id",
        "business_key",
        "event_ts",
        "feature_date",
        "decision_action",
        "aiops_severity",
        "primary_source_type",
        "reconstruction_error",
        "warning_threshold",
        "quarantine_threshold",
        "top_contributing_features_json",
        "scored_at",
    )
    .orderBy("layer", F.desc("reconstruction_error"))
)

display(warning_quarantine_df)

layer,record_id,business_key,event_ts,feature_date,decision_action,aiops_severity,primary_source_type,reconstruction_error,warning_threshold,quarantine_threshold,top_contributing_features_json,scored_at
gold,90_6085|1777435201|3001Z|2,90_6085|1777435201|3001Z|2,2026-04-29T04:00:01.251Z,2026-04-29,QUARANTINE,CRITICAL,validity,0.11288730055093765,0.0326591394841671,0.0979774184525013,"[{""feature"": ""longitude_clipped"", ""contribution"": 0.24097840487957, ""source_type"": ""validity""}, {""feature"": ""speed_clipped"", ""contribution"": 0.05464457347989082, ""source_type"": ""validity""}, {""feature"": ""latitude_clipped"", ""contribution"": 0.04303891584277153, ""source_type"": ""validity""}]",2026-04-29T05:44:20.814154Z
gold,90_6085|1777435200|3001Z|2,90_6085|1777435200|3001Z|2,2026-04-29T04:00:00.251Z,2026-04-29,QUARANTINE,CRITICAL,validity,0.11276188492774963,0.0326591394841671,0.0979774184525013,"[{""feature"": ""longitude_clipped"", ""contribution"": 0.24093928933143616, ""source_type"": ""validity""}, {""feature"": ""speed_clipped"", ""contribution"": 0.05463799834251404, ""source_type"": ""validity""}, {""feature"": ""latitude_clipped"", ""contribution"": 0.042708367109298706, ""source_type"": ""validity""}]",2026-04-29T05:44:20.814154Z
gold,90_6098|1777435201|3001Z|2,90_6098|1777435201|3001Z|2,2026-04-29T04:00:01.251Z,2026-04-29,QUARANTINE,CRITICAL,validity,0.11274979263544083,0.0326591394841671,0.0979774184525013,"[{""feature"": ""longitude_clipped"", ""contribution"": 0.240579292178154, ""source_type"": ""validity""}, {""feature"": ""speed_clipped"", ""contribution"": 0.05459330976009369, ""source_type"": ""validity""}, {""feature"": ""latitude_clipped"", ""contribution"": 0.043076787143945694, ""source_type"": ""validity""}]",2026-04-29T05:44:20.814154Z
gold,90_6051|1777435201|3001Z|2,90_6051|1777435201|3001Z|2,2026-04-29T04:00:01.254Z,2026-04-29,QUARANTINE,CRITICAL,validity,0.11260282248258591,0.0326591394841671,0.0979774184525013,"[{""feature"": ""longitude_clipped"", ""contribution"": 0.2401723861694336, ""source_type"": ""validity""}, {""feature"": ""speed_clipped"", ""contribution"": 0.05454006791114807, ""source_type"": ""validity""}, {""feature"": ""latitude_clipped"", ""contribution"": 0.04309600219130516, ""source_type"": ""validity""}]",2026-04-29T05:44:20.814154Z
gold,90_6098|1777435200|3001Z|2,90_6098|1777435200|3001Z|2,2026-04-29T04:00:00.251Z,2026-04-29,QUARANTINE,CRITICAL,validity,0.1126025840640068,0.0326591394841671,0.0979774184525013,"[{""feature"": ""longitude_clipped"", ""contribution"": 0.2405114322900772, ""source_type"": ""validity""}, {""feature"": ""speed_clipped"", ""contribution"": 0.054584063589572906, ""source_type"": ""validity""}, {""feature"": ""latitude_clipped"", ""contribution"": 0.0427122488617897, ""source_type"": ""validity""}]",2026-04-29T05:44:20.814154Z
gold,90_6085|1777435199|3001Z|2,90_6085|1777435199|3001Z|2,2026-04-29T03:59:59.25Z,2026-04-29,QUARANTINE,CRITICAL,validity,0.11258883029222488,0.0326591394841671,0.0979774184525013,"[{""feature"": ""longitude_clipped"", ""contribution"": 0.24083393812179565, ""source_type"": ""validity""}, {""feature"": ""speed_clipped"", ""contribution"": 0.05462272837758064, ""source_type"": ""validity""}, {""feature"": ""latitude_clipped"", ""contribution"": 0.04230980947613716, ""source_type"": ""validity""}]",2026-04-29T05:44:20.814154Z
gold,90_6098|1777435199|3001Z|2,90_6098|1777435199|3001Z|2,2026-04-29T03:59:59.251Z,2026-04-29,QUARANTINE,CRITICAL,validity,0.11246582120656967,0.0326591394841671,0.0979774184525013,"[{""feature"": ""longitude_clipped"", ""contribution"": 0.24045579135417938, ""source_type"": ""validity""}, {""feature"": ""speed_clipped"", ""contribution"": 0.0545755960047245, ""source_type"": ""validity""}, {""feature"": ""latitude_clipped"", ""contribution"": 0.042366065084934235, ""source_type"": ""validity""}]",2026-04-29T05:44:20.814154Z
gold,90_6051|1777435200|3001Z|2,90_6051|1777435200|3001Z|2,2026-04-29T04:00:00.2

## 5. Top Contributing Features

In [0]:
top_feature_schema = T.ArrayType(T.StructType([
    T.StructField("feature", T.StringType(), True),
    T.StructField("contribution", T.DoubleType(), True),
    T.StructField("source_type", T.StringType(), True),
]))

top_features_df = (
    aiops_decisions
    .filter(F.col("ingest_date") == F.current_date())
    .where(F.col("decision_action").isin("WARN", "QUARANTINE"))
    .select(
        "layer",
        "decision_action",
        "aiops_severity",
        F.explode_outer(F.from_json("top_contributing_features_json", top_feature_schema)).alias("feature_row"),
    )
    .where(F.col("feature_row").isNotNull())
    .select(
        "layer",
        "decision_action",
        "aiops_severity",
        F.col("feature_row.source_type").alias("source_type"),
        F.col("feature_row.feature").alias("feature"),
        F.col("feature_row.contribution").alias("contribution"),
    )
)

top_features_report_df = (
    top_features_df
    .groupBy("layer", "decision_action", "aiops_severity", "source_type", "feature")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.avg("contribution"), 8).alias("avg_contribution"),
        F.round(F.max("contribution"), 8).alias("max_contribution"),
    )
    .orderBy("layer", "decision_action", F.desc("record_count"), F.desc("avg_contribution"))
)

display(top_features_report_df)
save_report(top_features_report_df, REPORT_TABLES["top_features"])

layer,decision_action,aiops_severity,source_type,feature,record_count,avg_contribution,max_contribution
gold,QUARANTINE,CRITICAL,validity,longitude_clipped,114,0.23588786,0.2409784
gold,QUARANTINE,CRITICAL,validity,speed_clipped,114,0.05390918,0.05464457
gold,QUARANTINE,CRITICAL,validity,latitude_clipped,114,0.03414872,0.043096
gold,WARN,WARNING,validity,longitude_clipped,342,0.08922063,0.14151058
gold,WARN,WARNING,validity,latitude_clipped,342,0.03523021,0.05611475
gold,WARN,WARNING,validity,speed_clipped,342,0.00815557,0.09529996
silver,QUARANTINE,CRITICAL,validity,latitude_clipped,2040,54067.10345599,91608.53125
silver,QUARANTINE,CRITICAL,validity,speed_clipped,2040,24442.77692578,41414.546875
silver,QUARANTINE,CRITICAL,validity,heading_clipped,2040,12233.45570678,20727.33007813
silver,QUARANTINE,CRITICAL,validity,longitude_clipped,2040,5102.65496421,8645.31054688


{"saved_table": "`hant-catalog`.hsl.report_aiops_detection_top_features", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/reports/report_aiops_detection_top_features"}


## 6. Source Type Coverage From Contribution JSON

In [0]:
coverage_schema = T.MapType(T.StringType(), T.DoubleType())

source_type_coverage_df = (
    aiops_decisions
    .filter(F.col("ingest_date") == F.current_date())
    .where(F.col("decision_action").isin("WARN", "QUARANTINE"))
    .select(
        "layer",
        "decision_action",
        "aiops_severity",
        F.explode_outer(F.from_json("source_type_coverage_json", coverage_schema)).alias("source_type", "contribution"),
    )
    .where(F.col("source_type").isNotNull())
    .groupBy("layer", "decision_action", "aiops_severity", "source_type")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.sum("contribution"), 8).alias("total_contribution"),
        F.round(F.avg("contribution"), 8).alias("avg_contribution"),
    )
    .orderBy("layer", "decision_action", F.desc("total_contribution"))
)

display(source_type_coverage_df)

layer,decision_action,aiops_severity,source_type,record_count,total_contribution,avg_contribution
gold,QUARANTINE,CRITICAL,validity,114,36.92981717,0.32394576
gold,WARN,WARNING,validity,342,45.35139467,0.13260642
silver,QUARANTINE,CRITICAL,validity,2040,1.9649903541865313E8,96323.05657777
silver,QUARANTINE,CRITICAL,timeliness,2040,290383.38608748,142.3447971
silver,WARN,WARNING,validity,9906,5785.98305262,0.58408874
silver,WARN,WARNING,timeliness,9906,471.53603383,0.04760105


## 7. Runtime Summary

In [0]:
if table_exists(RUNTIME_TABLE):
    runtime_df = spark.table(qname(RUNTIME_TABLE))
    if "runtime_date" in runtime_df.columns:
        runtime_df = runtime_df.where(F.col("runtime_date") == F.current_date())

    runtime_summary_df = (
        runtime_df
        .where(F.col("runtime_stage") == "aiops_record_score")
        .groupBy("layer", "model_version")
        .agg(
            F.count("*").alias("microbatch_count"),
            F.sum("input_record_count").cast("bigint").alias("input_records"),
            F.sum("scored_record_count").cast("bigint").alias("scored_records"),
            F.round(F.sum("runtime_ms") / 1000.0, 2).alias("total_runtime_sec"),
            F.round(F.avg("runtime_ms"), 2).alias("avg_microbatch_runtime_ms"),
            F.round(F.max("runtime_ms"), 2).alias("max_microbatch_runtime_ms"),
        )
        .withColumn(
            "records_per_second",
            F.when(F.col("total_runtime_sec") > 0, F.round(F.col("scored_records") / F.col("total_runtime_sec"), 2))
        )
        .orderBy("layer", "model_version")
    )
    display(runtime_summary_df)
    save_report(runtime_summary_df, REPORT_TABLES["runtime"])
else:
    print(f"Runtime table not found: {qname(RUNTIME_TABLE)}")

layer,model_version,microbatch_count,input_records,scored_records,total_runtime_sec,avg_microbatch_runtime_ms,max_microbatch_runtime_ms,records_per_second
bronze,2026-04-28,1,142385,142385,297.04,297041.57,297041.57,479.35
gold,2026-04-28,1,126240,126240,366.38,366376.57,366376.57,344.56
silver,2026-04-28,1,128280,128280,334.04,334043.2,334043.2,384.03


{"saved_table": "`hant-catalog`.hsl.report_aiops_detection_runtime_summary", "path": "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/reports/report_aiops_detection_runtime_summary"}


## 8. Compact Text Summary

In [0]:
summary_rows = summary_df.collect()
for row in summary_rows:
    print(
        f"{row['layer'].upper()}: "
        f"total={row['total_scored_records']}, "
        f"PASS={row['pass_records']} ({row['pass_pct']}%), "
        f"WARN={row['warning_records']} ({row['warning_pct']}%), "
        f"QUARANTINE={row['quarantine_records']} ({row['quarantine_pct']}%), "
        f"avg_error={row['avg_reconstruction_error']}, "
        f"max_error={row['max_reconstruction_error']}"
    )

BRONZE: total=142385, PASS=142385 (100.0%), WARN=0 (0.0%), QUARANTINE=0 (0.0%), avg_error=1.742e-05, max_error=2.068e-05
GOLD: total=126240, PASS=125784 (99.64%), WARN=342 (0.27%), QUARANTINE=114 (0.09%), avg_error=0.00272462, max_error=0.1128873
SILVER: total=128280, PASS=116334 (90.69%), WARN=9906 (7.72%), QUARANTINE=2040 (1.59%), avg_error=255.71271022, max_error=27240.72070313
